In [167]:
import torch
import torch.nn as nn
from torch.nn.parameter import Parameter

#Experimentation with classes, ANFIS layers, operations,...



##0 Parameters
These classes were made as an attemp to fix a problem with gradients calculation.

In [2]:
class Premises(nn.Module):
    '''
    Class that represents the premises parameters of an ANFIS

    To initialize it:
        x_train: training data set; this parameter its used to initialize the premises uniformly,
                 if it is not entered, the premises will be initialized randomly
    '''
    def __init__(self, x_train=[]):
        super(Premises, self).__init__()
        if x_train == []:
            self.premises = torch.rand(input_size, rules, len(params))
        else:
            self.premises = torch.rand(input_size, rules, len(params))
            if self.rules > 1:
                min = torch.min(x_train, dim=0).values
                max = torch.max(x_train, dim=0).values
                stp = (max - min) / (self.rules - 1)
                for i in range(self.input_size):
                    h = torch.arange(min[i], max[i] + stp[i], stp[i])
                    for j in range(self.rules):
                        self.premises[i][j][0] = h[j]
                        self.premises[i][j][1] = stp[i]/2
                        if (len(self.params) == 3): self.premises[i][j][2] = 2
            else:
                for i in range(self.input_size):
                    for j in range(self.rules):
                        self.premises[i][j][0] = torch.std(x_train[:, i])
                        self.premises[i][j][1] = torch.mean(x_train[:, i])
                        if (len(self.params) == 3): self.premises[i][j][2] = 2
        self.premises.requires_grad = True

In [3]:
class Consequents(nn.Module):
    '''
    Class that represents the consequents parameters of an ANFIS
    '''
    def __init__(self, x_train=[]):
        super(Consequents, self).__init__()
        self.premises = torch.rand(input_size, rules, len(params), requires_grad = True)

##1 Fuzzification Layer


###1.1 Proposal 1
To initialize it:
* data_shape: Shape of a batch of data
* rules : Number of initial rules (default = 1)
* mf: Function to be used as MemberShip Function. This function must be written in such a way that it receives a tensor as parameters (default = gaussian_3, Gaussian function with 3 parameters)
* params : List with the names of the MF parameters to use (default = ['sigma', 'mu', 'f'])
* init_mode: Premises initialization mode (default = 0)
  
The idea is that the layer stores the premises in a tensor that will have as an attribute, this implies that the mf that you want to apply must be written in a special way: 2 parameters, x (matrix of input data) and p (matrix of parameters to be used for each input data).

In [4]:
def gaussian(x, mu, sigma, f=1):
    return torch.exp( -f * torch.pow((x - mu), 2) / (2*torch.pow(sigma, 2)) )

In [5]:
def gaussian_3(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [6]:
unic_tensor = torch.tensor([1, 2])
x_multi = torch.tensor([[1, 2],
                        [2, 4],
                        [3, 6]])

In [7]:
def gaussian_2(x, p):
    return torch.exp(-torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [8]:
params=['sigma', 'mu', 'f']
rules = 3
shape = x_multi.shape
shape

torch.Size([3, 2])

In [9]:
x_multi.unsqueeze(2).repeat(1, 1, rules)

tensor([[[1, 1, 1],
         [2, 2, 2]],

        [[2, 2, 2],
         [4, 4, 4]],

        [[3, 3, 3],
         [6, 6, 6]]])

In [10]:
premises = torch.rand(shape[1], rules, len(params))
#                    (       2,        3,           3)
premises

tensor([[[0.4949, 0.9332, 0.5479],
         [0.0713, 0.7423, 0.7865],
         [0.3448, 0.7791, 0.0966]],

        [[0.8280, 0.0633, 0.5786],
         [0.2693, 0.6450, 0.2334],
         [0.7454, 0.9832, 0.4727]]])

In [11]:
y = gaussian_3(x_multi.unsqueeze(2).repeat(1, 1, rules), premises)
y

tensor([[[9.2287e-01, 5.4037e-01, 9.6643e-01],
         [9.5288e-44, 4.3151e-01, 6.8057e-01]],

        [[4.9034e-01, 7.0327e-02, 8.0416e-01],
         [0.0000e+00, 2.0135e-02, 7.5045e-02]],

        [[1.3887e-01, 2.1964e-03, 5.7071e-01],
         [0.0000e+00, 9.9545e-05, 1.1705e-03]]])

In [12]:
class FuzzifyLayer(nn.Module):
    '''
    Class that represents the first layer (fuzzification layer) of an ANFIS

    Properties:
        data_shape : dimension of the input data
        rules : number of initial rules
        mf: function used as membership function
        params: list with parameter names of the mf
        premises: array with the parameters used in each node of this layer
    '''
    def __init__(self, data_shape, rules=3, mf=gaussian_3, params=['sigma', 'mu', 'f'], init_mode=0):
        super(FuzzifyLayer, self).__init__()
        self.rules = rules
        self.data_shape = data_shape
        self.mf = mf
        self.params = params
        if init_mode == 0:
            self.premises = torch.rand(data_shape[1], rules, len(params))
        elif init_mode == 1:
            #ANOTHER WAY TO INITIALIZE THE PREMISES
            self.premises = torch.ones(data_shape[1], rules, len(params))

    def forward(self, x):
        return self.mf(x.unsqueeze(2).repeat(1, 1, self.rules), self.premises)

    def show_premises(self):
        print("Premises tensor:")
        print(self.premises)

    def premises(self):
        return self.premises

In [13]:
train_data = torch.rand(5, 2)
train_data #5 2-dimensional input vectors (batch size = 5)

tensor([[0.5234, 0.3513],
        [0.5379, 0.0069],
        [0.8262, 0.0169],
        [0.0253, 0.3757],
        [0.3157, 0.2126]])

In [14]:
fuzzify_layer = FuzzifyLayer(train_data.shape, rules=3)

In [15]:
'''
x1 premises
[[[mu, sigma, f],   (feature 1)
  [mu, sigma, f],   (feature 2)
  [mu, sigma, f]],  (feature 3)

x2 premises
 [[mu, sigma, f],   (feature 1)
  [mu, sigma, f],   (feature 2)
  [mu, sigma, f]]]  (feature 3)
'''
fuzzify_layer.show_premises()

Premises tensor:
tensor([[[0.0210, 0.6570, 0.1045],
         [0.0488, 0.6061, 0.7047],
         [0.7217, 0.5796, 0.2265]],

        [[0.5729, 0.5430, 0.0553],
         [0.3655, 0.9860, 0.1448],
         [0.6832, 0.5432, 0.0300]]])


In [16]:
output1 = fuzzify_layer(train_data)
'''
batch input 1
[[[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]],  (x2)

batch input 2
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)

...
...

last batch input
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)
'''
output1

tensor([[[0.9699, 0.8057, 0.9868],
         [0.9954, 1.0000, 0.9944]],

        [[0.9682, 0.7950, 0.9887],
         [0.9704, 0.9905, 0.9770]],

        [[0.9245, 0.5601, 0.9963],
         [0.9714, 0.9910, 0.9777]],

        [[1.0000, 0.9995, 0.8492],
         [0.9964, 1.0000, 0.9952]],

        [[0.9895, 0.9340, 0.9459],
         [0.9879, 0.9983, 0.9888]]])

###1.1.O Proposal 1 Optimized and Flexible
The idea is to only use pytorch functions and allow operations for both an input data and a batch of data.

In [17]:
def gaussian_3a(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x.unsqueeze(x.dim()) - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [18]:
def gaussian_3b(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

#input is required to be x.unsqueeze(x.dim())

In [19]:
uniq_tensor = torch.tensor([1, 2])
x_multi = torch.tensor([[1, 2],
                        [2, 4],
                        [3, 6]])

In [20]:
premises = torch.tensor([[[10, 20, 1],
                          [30, 40, 2]],

                         [[50, 60, 3],
                          [70, 80, 4]]])

In [21]:
gaussian_3a(uniq_tensor, premises)

tensor([[0.9037, 0.5912],
        [0.3829, 0.2357]])

In [22]:
gaussian_3a(x_multi, premises)

tensor([[[0.9037, 0.5912],
         [0.3829, 0.2357]],

        [[0.9231, 0.6126],
         [0.4141, 0.2563]],

        [[0.9406, 0.6341],
         [0.4463, 0.2780]]])

De esta manera es posible definir diversas funciones para usarlas como MFs:

In [23]:
uniq_tensor = torch.tensor([0.4681, 0.3757])
uniq_tensor

tensor([0.4681, 0.3757])

In [24]:
x_multi = torch.tensor([[0.4681, 0.3757],
        [0.6850, 0.1338],
        [0.9057, 0.2981]])
x_multi

tensor([[0.4681, 0.3757],
        [0.6850, 0.1338],
        [0.9057, 0.2981]])

In [25]:
#For functions with 4 parameters
premises4 = torch.rand(2, 2, 4)
premises4

tensor([[[6.2283e-01, 1.7086e-01, 5.1565e-01, 6.7544e-02],
         [8.8223e-01, 3.0459e-01, 8.2637e-02, 3.2687e-04]],

        [[8.7512e-01, 3.9028e-01, 6.9087e-01, 5.6238e-01],
         [7.0383e-01, 1.8940e-01, 8.1075e-01, 2.1717e-01]]])

In [26]:
#For functions with 3 parameters
premises3 = torch.rand(2, 2, 3)
premises3

tensor([[[0.0504, 0.5810, 0.0725],
         [0.1057, 0.6628, 0.3720]],

        [[0.5305, 0.6122, 0.4117],
         [0.9149, 0.7641, 0.5265]]])

In [27]:
#For functions with 2 parameters
premises2 = torch.rand(2, 2, 2)
premises2

tensor([[[0.6029, 0.7795],
         [0.2066, 0.0626]],

        [[0.1519, 0.5321],
         [0.0204, 0.2897]]])

In [28]:
#INPUT x WILL BE CONSIDERED TO ALWAYS BE: x.unsqueeze(x.dim())

In [29]:
gaussian2_params = ['sigma', 'mu']
def gaussian2(x, p):
    return torch.exp(torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [30]:
gaussian2(uniq_tensor.unsqueeze(unic_tensor.dim()), premises2)

tensor([[1.0151e+00, 6.2270e+03],
        [1.0925e+00, 2.1218e+00]])

In [31]:
gaussian2(x_multi.unsqueeze(x_multi.dim()), premises2)

tensor([[[1.0151e+00, 6.2270e+03],
         [1.0925e+00, 2.1218e+00]],

        [[1.0056e+00, 4.9864e+12],
         [1.0006e+00, 1.0796e+00]],

        [[1.0783e+00, 1.3030e+27],
         [1.0385e+00, 1.5834e+00]]])

In [32]:
gaussian3_params = ['sigma', 'mu', 'f']
def gaussian3(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [33]:
gaussian3(uniq_tensor.unsqueeze(unic_tensor.dim()), premises3)

tensor([[0.9814, 0.9459],
        [0.9869, 0.8772]])

In [34]:
gaussian3(x_multi.unsqueeze(x_multi.dim()), premises3)

tensor([[[0.9814, 0.9459],
         [0.9869, 0.8772]],

        [[0.9577, 0.8676],
         [0.9172, 0.7595]],

        [[0.9244, 0.7627],
         [0.9708, 0.8424]]])

In [35]:
bell_params = ["a", "b", "c"]
def bell(x, p):
    return 1 / (1 + torch.pow(torch.abs((x - p[:, :, 2]) / p[:, :, 0]), 2 * p[:, :, 1]))

In [36]:
bell(uniq_tensor.unsqueeze(unic_tensor.dim()), premises3)

tensor([[0.0836, 0.5316],
        [0.9642, 0.9402]])

In [37]:
bell(x_multi.unsqueeze(x_multi.dim()), premises3)

tensor([[[0.0836, 0.5316],
         [0.9642, 0.9402]],

        [[0.0521, 0.1917],
         [0.6882, 0.7846]],

        [[0.0370, 0.1047],
         [0.8684, 0.8929]]])

In [38]:
triangular_params = ["a", "b"]
def triangular(x, p):
    return torch.clamp(1 - torch.abs((x - p[:, :, 1]) / p[:, :, 0]), min=0)

In [39]:
triangular(uniq_tensor.unsqueeze(unic_tensor.dim()), premises2)

tensor([[0.4835, 0.0000],
        [0.0000, 0.0000]])

In [40]:
triangular(x_multi.unsqueeze(x_multi.dim()), premises2)

tensor([[[0.4835, 0.0000],
         [0.0000, 0.0000]],

        [[0.8432, 0.0000],
         [0.0000, 0.0000]],

        [[0.7908, 0.0000],
         [0.0000, 0.5862]]])

In [41]:
trapezoidal_params = ["a", "b", "c", "d"]
def trapezoidal(x, p):
    return torch.min(torch.clamp((x - p[:, :, 0]) / (p[:, :, 1] - p[:, :, 0]), min=0, max=1), torch.clamp((p[:, :, 3] - x) / (p[:, :, 3] - p[:, :, 2]), min=0, max=1))

In [42]:
trapezoidal(uniq_tensor.unsqueeze(unic_tensor.dim()), premises4)

tensor([[0.3423, 0.7169],
        [0.0000, 0.2671]])

In [43]:
trapezoidal(x_multi.unsqueeze(x_multi.dim()), premises4)

tensor([[[0.3423, 0.7169],
         [0.0000, 0.2671]],

        [[0.0000, 0.3414],
         [0.0000, 0.0000]],

        [[0.0000, 0.0000],
         [0.0000, 0.1363]]])

Ahora ya se tienen funciones que pueden usarse como MFs flexibles (permiten operaciones sobre un dato de entrada solo y también sobre batches de estos mismos)

In [44]:
class FuzzifyLayer(nn.Module):
    '''
    Class that represents the first layer (fuzzification layer) of an ANFIS

    Attributes:
        input_size : size of a single input
        rules : number of initial rules
        mf: function used as membership function
        params: list with parameter names
        premises: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_size, rules=3, mf=gaussian3, params=['sigma', 'mu', 'f'], init_mode=0):
        super(FuzzifyLayer, self).__init__()
        self.input_size = input_size
        self.rules = rules
        self.mf = mf
        self.params = params
        if init_mode == 0:
            self.premises = torch.rand(input_size, rules, len(params))
        elif init_mode == 1:
            #ANOTHER WAY TO INITIALIZE THE PREMISES
            self.premises = torch.ones(input_size, rules, len(params))

    def forward(self, x):
        return self.mf(x.unsqueeze(x.dim()), self.premises)

    @property
    def premises_structure(self):
        print("Premises Structure:")
        for i in range(self.input_size):
            print(f"    x{i} parameters:")
            for j in range(self.rules):
                print(f"        feature {j + 1}:")
                [print(f"            {self.params[k]}: {self.premises[i, j, k]}") for k in range(len(self.params))]


In [45]:
layer1_0 = FuzzifyLayer(x_multi.shape[1])

In [46]:
print("Premises: ")
layer1_0.premises

Premises: 


tensor([[[0.0862, 0.9214, 0.0666],
         [0.3654, 0.8526, 0.0153],
         [0.6765, 0.6578, 0.0472]],

        [[0.2828, 0.2006, 0.7995],
         [0.1957, 0.4862, 0.5037],
         [0.2916, 0.0358, 0.5563]]])

In [47]:
layer1_0.premises_structure

Premises Structure:
    x0 parameters:
        feature 1:
            sigma: 0.08624821901321411
            mu: 0.9214279055595398
            f: 0.06658703088760376
        feature 2:
            sigma: 0.36537379026412964
            mu: 0.8525872230529785
            f: 0.015292823314666748
        feature 3:
            sigma: 0.6764623522758484
            mu: 0.6577757000923157
            f: 0.04719144105911255
    x1 parameters:
        feature 1:
            sigma: 0.28283941745758057
            mu: 0.2005944848060608
            f: 0.7995227575302124
        feature 2:
            sigma: 0.1956530213356018
            mu: 0.4862452745437622
            f: 0.5037283301353455
        feature 3:
            sigma: 0.29156559705734253
            mu: 0.03583192825317383
            f: 0.5562854409217834


In [48]:
batch_data0 = torch.rand(5,2)
batch_data0

tensor([[0.6453, 0.5495],
        [0.1682, 0.9843],
        [0.2054, 0.4795],
        [0.0258, 0.0656],
        [0.7770, 0.1938]])

In [49]:
single_data0 = torch.rand(2)
single_data0

tensor([0.0554, 0.3409])

In [50]:
batch_output1_0 = layer1_0(batch_data0)
'''
batch input 1
[[[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]],  (x2)

batch input 2
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)

...
...

last batch input
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)
'''
batch_output1_0

tensor([[[9.8782e-01, 9.9918e-01, 9.9995e-01],
         [4.9335e-01, 8.7512e-01, 5.4940e-07]],

        [[9.9974e-01, 9.9959e-01, 9.8601e-01],
         [7.5302e-03, 5.1551e-01, 1.4013e-45]],

        [[9.9944e-01, 9.9973e-01, 9.8797e-01],
         [6.8097e-01, 9.1775e-01, 4.7530e-04]],

        [[9.9986e-01, 9.9879e-01, 9.7718e-01],
         [6.2582e-01, 9.8215e-01, 1.5764e-05]],

        [[9.8146e-01, 9.9822e-01, 9.9945e-01],
         [9.2431e-01, 1.0000e+00, 1.2627e-01]]])

In [51]:
single_output1_0 = layer1_0(single_data0)
'''
[[mf(feature1), mf(feature2), mf(feature3)],   (x1)
 [mf(feature1), mf(feature2), mf(feature3)]]   (x2)
'''
single_output1_0

tensor([[1.0000, 0.9990, 0.9792],
        [0.9671, 0.9778, 0.5906]])

###1.1.2 Proposal 1 with initialization of MATLAB premises

In [52]:
gaussian3_params = ['sigma', 'mu', 'f']
def gaussian3(x, p):
    return torch.exp(-p[:, :, 2] * torch.pow((x - p[:, :, 0]), 2) / (2 * torch.pow(p[:, :, 1], 2)))

In [53]:
gaussian3_params = ['sigma', 'mu', 'f']
def gaussian3_matlab(x, p):
    return torch.exp(-p[:, :, 2] * (torch.pow((x - p[:, :, 0]) / p[:, :, 1], 2)))

In [54]:
gaussian2_params = ['sigma', 'mu']
def gaussian2(x, p):
    return torch.exp(-(torch.pow((x - p[:, :, 0]) / p[:, :, 1], 2)))

In [55]:
batch_data1 = torch.tensor([[0.6294, 0.2647, 0.9150, 0.9143, -0.1565],
   [0.8116, -0.8049, 0.9298, -0.0292, 0.8315],
   [-0.7460, -0.4430, -0.6848, 0.6006, 0.5844],
   [0.8268, 0.0938, 0.9412, -0.7162, 0.9190]])
batch_data1

tensor([[ 0.6294,  0.2647,  0.9150,  0.9143, -0.1565],
        [ 0.8116, -0.8049,  0.9298, -0.0292,  0.8315],
        [-0.7460, -0.4430, -0.6848,  0.6006,  0.5844],
        [ 0.8268,  0.0938,  0.9412, -0.7162,  0.9190]])

In [56]:
mi = torch.min(batch_data1, dim=0).values
mi

tensor([-0.7460, -0.8049, -0.6848, -0.7162, -0.1565])

In [57]:
ma = torch.max(batch_data1, dim=0).values
ma

tensor([0.8268, 0.2647, 0.9412, 0.9143, 0.9190])

In [58]:
ex = ma - mi
ex

tensor([1.5728, 1.0696, 1.6260, 1.6305, 1.0755])

In [59]:
rules = 3
dim = 5

In [60]:
stp = ex / (rules - 1)
stp

tensor([0.7864, 0.5348, 0.8130, 0.8153, 0.5378])

In [61]:
h1 = torch.arange(mi[0], ma[0] + stp[0], stp[0])
h1

tensor([-0.7460,  0.0404,  0.8268,  1.6132])

In [62]:
h2 = torch.arange(mi[1], ma[1] + stp[1], stp[1])
h2

tensor([-0.8049, -0.2701,  0.2647])

In [63]:
h3 = torch.arange(mi[2], ma[2] + stp[2], stp[2])
h3

tensor([-0.6848,  0.1282,  0.9412])

In [64]:
h4 = torch.arange(mi[3], ma[3] + stp[3], stp[3])
h4

tensor([-0.7162,  0.0991,  0.9143])

In [65]:
h5 = torch.arange(mi[4], ma[4] + stp[4], stp[4])
h5

tensor([-0.1565,  0.3813,  0.9190,  1.4568])

In [66]:
pp = torch.rand(dim, rules, 3)
pp

tensor([[[0.9370, 0.0549, 0.3903],
         [0.2393, 0.7778, 0.0631],
         [0.6256, 0.0644, 0.8091]],

        [[0.1972, 0.5465, 0.9261],
         [0.3366, 0.5429, 0.3732],
         [0.8882, 0.1068, 0.7681]],

        [[0.6683, 0.0083, 0.2721],
         [0.8817, 0.3897, 0.1638],
         [0.1405, 0.8552, 0.0410]],

        [[0.5140, 0.5042, 0.5215],
         [0.4711, 0.7165, 0.7883],
         [0.0818, 0.8563, 0.5195]],

        [[0.3492, 0.3947, 0.0614],
         [0.4556, 0.5254, 0.9111],
         [0.3336, 0.3773, 0.2336]]])

In [67]:
pp[0][1][2]

tensor(0.0631)

In [68]:
pp = torch.zeros(dim, rules, 3)
for i in range(dim):
    h = torch.arange(mi[i], ma[i] + stp[i], stp[i])
    for j in range(rules):
        pp[i][j][0] = h[j]
        pp[i][j][1] = stp[i]/2
        pp[i][j][2] = 2
pp

tensor([[[-0.7460,  0.3932,  2.0000],
         [ 0.0404,  0.3932,  2.0000],
         [ 0.8268,  0.3932,  2.0000]],

        [[-0.8049,  0.2674,  2.0000],
         [-0.2701,  0.2674,  2.0000],
         [ 0.2647,  0.2674,  2.0000]],

        [[-0.6848,  0.4065,  2.0000],
         [ 0.1282,  0.4065,  2.0000],
         [ 0.9412,  0.4065,  2.0000]],

        [[-0.7162,  0.4076,  2.0000],
         [ 0.0991,  0.4076,  2.0000],
         [ 0.9143,  0.4076,  2.0000]],

        [[-0.1565,  0.2689,  2.0000],
         [ 0.3813,  0.2689,  2.0000],
         [ 0.9190,  0.2689,  2.0000]]])

In [69]:
torch.std(batch_data1[:, 0])

tensor(0.7563)

In [70]:
torch.mean(batch_data1[:, 0])

tensor(0.3805)

In [71]:
pp0 = torch.zeros(dim, 1, 3)
for i in range(dim):
    for j in range(1):
        pp0[i][j][0] = torch.std(batch_data1[:, i])
        pp0[i][j][1] = torch.mean(batch_data1[:, i])
        pp0[i][j][2] = 2
pp0

tensor([[[ 0.7563,  0.3805,  2.0000]],

        [[ 0.4917, -0.2223,  2.0000]],

        [[ 0.8068,  0.5253,  2.0000]],

        [[ 0.7217,  0.1924,  2.0000]],

        [[ 0.4884,  0.5446,  2.0000]]])

In [72]:
class FuzzifyLayer(nn.Module):
    '''
    Class that represents the first layer (fuzzification layer) of an ANFIS

    Attributes:
        input_size : size of a single input
        rules : number of initial rules
        mf: function used as membership function
        params: list with parameter names
        premises: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_size, rules=3, mf=gaussian3, params=['sigma', 'mu', 'f']):
        super(FuzzifyLayer, self).__init__()
        self.input_size = input_size
        self.rules = rules
        self.mf = mf
        self.params = params
        self.premises = torch.rand(input_size, rules, len(params))

    def forward(self, x):
        return self.mf(x.unsqueeze(x.dim()), self.premises)

    #For both gaussian_3 and gaussian_2 function
    def uniform_premises(self, x_train):
        if self.rules > 1:
            min = torch.min(x_train, dim=0).values
            max = torch.max(x_train, dim=0).values
            stp = (max - min) / (self.rules - 1)
            for i in range(self.input_size):
                h = torch.arange(mi[i], ma[i] + stp[i], stp[i])
                for j in range(self.rules):
                    self.premises[i][j][0] = h[j]
                    self.premises[i][j][1] = stp[i]/2
                    if (len(self.params) == 3): self.premises[i][j][2] = 2
        else:
            for i in range(self.input_size):
                for j in range(self.rules):
                    self.premises[i][j][0] = torch.std(x_train[:, i])
                    self.premises[i][j][1] = torch.mean(x_train[:, i])
                    if (len(self.params) == 3): self.premises[i][j][2] = 2

    @property
    def premises_structure(self):
        print("Premises Structure:")
        for i in range(self.input_size):
            print(f"    x{i} parameters:")
            for j in range(self.rules):
                print(f"        rule {j + 1}:")
                [print(f"            {self.params[k]}: {self.premises[i, j, k]}") for k in range(len(self.params))]

In [73]:
layer1_matlab = FuzzifyLayer(batch_data1.shape[1])
layer1_matlab.uniform_premises(batch_data1)

In [74]:
layer1_matlab.premises_structure

Premises Structure:
    x0 parameters:
        rule 1:
            sigma: -0.7459999918937683
            mu: 0.39319998025894165
            f: 2.0
        rule 2:
            sigma: 0.04039996862411499
            mu: 0.39319998025894165
            f: 2.0
        rule 3:
            sigma: 0.8267999291419983
            mu: 0.39319998025894165
            f: 2.0
    x1 parameters:
        rule 1:
            sigma: -0.8048999905586243
            mu: 0.26739999651908875
            f: 2.0
        rule 2:
            sigma: -0.2700999975204468
            mu: 0.26739999651908875
            f: 2.0
        rule 3:
            sigma: 0.2646999955177307
            mu: 0.26739999651908875
            f: 2.0
    x2 parameters:
        rule 1:
            sigma: -0.6848000288009644
            mu: 0.4065000116825104
            f: 2.0
        rule 2:
            sigma: 0.1281999945640564
            mu: 0.4065000116825104
            f: 2.0
        rule 3:
            sigma: 0.94120001792

In [75]:
output_1_matlab = layer1_matlab(batch_data1)

###1.P Some things to consider
* **COULD IT BE BETTER TO DEFINE THE GAUSSIAN FUNCTION WITHIN THE LAYER?** (I don't think so, the model would lose flexibility)
* **MATLAB PREMISES INITIALIZATION STILL MUST BE APPLIED**

###1.2 Proposal 2

In [76]:
#STILL NOTHING

##2 Middle Layers

###2.1 Generalized "AND" Layer

In [77]:
def multiplication_and(tensor):
    return tensor.transpose(1, 2).prod(dim=2)

In [78]:
result = multiplication_and(output1)
result

tensor([[0.9654, 0.8057, 0.9813],
        [0.9395, 0.7874, 0.9659],
        [0.8981, 0.5550, 0.9741],
        [0.9964, 0.9995, 0.8451],
        [0.9776, 0.9323, 0.9354]])

In [79]:
class GeneralizedANDLayer(nn.Module):
    '''
    Class that represents the second layer of an ANFIS
    '''
    def __init__(self):
        super(GeneralizedANDLayer, self).__init__()

    def forward(self, x):
        return x.transpose(1, 2).prod(dim=2)

In [80]:
class GeneralizedANDLayer2(nn.Module):
    '''
    Class that represents the second layer of an ANFIS

    Propertues:
        and_function: function that is used as an AND operator
    '''
    def __init__(self, and_function=multiplication_and):
        super(GeneralizedANDLayer2, self).__init__()
        self.and_function = and_function

    def forward(self, x):
        return self.and_function(x)

In [81]:
generalizedAND_layer = GeneralizedANDLayer2()

In [82]:
output2 = generalizedAND_layer(output1)
'''
[[w1, w2, w3],  (of batch input1)
 [w1, w2, w3],  (of batch input2)
 ...
 [w1, w2, w3]]  (of last batch input)
'''
output2

tensor([[0.9654, 0.8057, 0.9813],
        [0.9395, 0.7874, 0.9659],
        [0.8981, 0.5550, 0.9741],
        [0.9964, 0.9995, 0.8451],
        [0.9776, 0.9323, 0.9354]])

###2.2 Normalization Layer


In [83]:
tensor_input = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32)
normalized_tensor = torch.nn.functional.normalize(tensor_input)
print(normalized_tensor)

tensor([[0.2673, 0.5345, 0.8018],
        [0.4558, 0.5698, 0.6838]])


In [84]:
class NormalizationLayer(nn.Module):
    '''
    Class that represents the third layer of an ANFIS
    '''
    def __init__(self):
        super(NormalizationLayer, self).__init__()

    def forward(self, x):
        return torch.nn.functional.normalize(x)

In [85]:
normalization_layer = NormalizationLayer()

In [86]:
output3 = normalization_layer(output2)
''' ~w = normalization(w)
[[~w1, ~w2, ~w3],  (of batch input1)
 [~w1, ~w2, ~w3],  (of batch input2)
 ...
 [~w1, ~w2, ~w3]]  (of last batch input)
'''
output3

tensor([[0.6053, 0.5051, 0.6152],
        [0.6020, 0.5045, 0.6189],
        [0.6252, 0.3864, 0.6781],
        [0.6057, 0.6076, 0.5138],
        [0.5950, 0.5674, 0.5693]])

###2.3 Generalized "AND" + Normalization Layer (Normalized Firing levels layer)
The idea is to do the AND and normalization operation in a single layer (since there are no trainable parameters here), in this way the outputs would be the already normalized firing levels

In [87]:
def multiplication_and(tensor):
    return tensor.transpose(1, 2).prod(dim=2)

In [88]:
class FiringLevelsLayer(nn.Module):
    '''
    Class that represents the second layer of an ANFIS

    Attributes:
        and_function: function that is used as an AND operator
    '''
    def __init__(self, and_function=multiplication_and):
        super(FiringLevelsLayer, self).__init__()
        self.and_function = and_function

    def forward(self, x):
        return torch.nn.functional.normalize(self.and_function(x))

In [89]:
firing_levels_layer = FiringLevelsLayer()

In [90]:
output23 = firing_levels_layer(output1)
''' ~w = normalization(w)
[[~w1, ~w2, ~w3],  (of batch input1)
 [~w1, ~w2, ~w3],  (of batch input2)
 ...
 [~w1, ~w2, ~w3]]  (of last batch input)
'''
output23

tensor([[0.6053, 0.5051, 0.6152],
        [0.6020, 0.5045, 0.6189],
        [0.6252, 0.3864, 0.6781],
        [0.6057, 0.6076, 0.5138],
        [0.5950, 0.5674, 0.5693]])

###2.3.O Normalized Firing Levels Layer Optimized and Flexible
The idea is to only use pytorch functions and allow operations for both an input data and a batch of data.

In [91]:
single_output1_0 = layer1_0(single_data0)
'''
[[mf(feature1), mf(feature2), mf(feature3)],   (x1)
 [mf(feature1), mf(feature2), mf(feature3)]]   (x2)
'''
single_output1_0

tensor([[1.0000, 0.9990, 0.9792],
        [0.9671, 0.9778, 0.5906]])

In [92]:
batch_output1_0 = layer1_0(batch_data0)
'''
batch input1
[[[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]],  (x2)

batch input2
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)

...
...

last batch input
 [[mf(feature1), mf(feature2), mf(feature3)],   (x1)
  [mf(feature1), mf(feature2), mf(feature3)]]   (x2)
'''
batch_output1_0

tensor([[[9.8782e-01, 9.9918e-01, 9.9995e-01],
         [4.9335e-01, 8.7512e-01, 5.4940e-07]],

        [[9.9974e-01, 9.9959e-01, 9.8601e-01],
         [7.5302e-03, 5.1551e-01, 1.4013e-45]],

        [[9.9944e-01, 9.9973e-01, 9.8797e-01],
         [6.8097e-01, 9.1775e-01, 4.7530e-04]],

        [[9.9986e-01, 9.9879e-01, 9.7718e-01],
         [6.2582e-01, 9.8215e-01, 1.5764e-05]],

        [[9.8146e-01, 9.9822e-01, 9.9945e-01],
         [9.2431e-01, 1.0000e+00, 1.2627e-01]]])

In [93]:
batch_output1_0.dim()

3

In [94]:
single_output1_0.dim()

2

In [95]:
def multiplication_and(x):
    return x.prod(dim=x.dim()-2)

In [96]:
batch_firing_levels = multiplication_and(batch_output1_0)
''' wi = feature [i] firing level
[[w1, w2, w3],  (of batch input1)
 [w1, w2, w3],  (of batch input2)
 ...
 [w1, w2, w3]]  (of last batch input)
'''
batch_firing_levels

tensor([[4.8734e-01, 8.7440e-01, 5.4937e-07],
        [7.5283e-03, 5.1530e-01, 1.4013e-45],
        [6.8059e-01, 9.1750e-01, 4.6958e-04],
        [6.2573e-01, 9.8096e-01, 1.5405e-05],
        [9.0717e-01, 9.9822e-01, 1.2620e-01]])

In [97]:
batch_firing_levels.dim()

2

In [98]:
torch.nn.functional.normalize(batch_firing_levels, p=1, dim=-1)

tensor([[3.5788e-01, 6.4212e-01, 4.0343e-07],
        [1.4399e-02, 9.8560e-01, 2.8026e-45],
        [4.2575e-01, 5.7396e-01, 2.9375e-04],
        [3.8945e-01, 6.1054e-01, 9.5877e-06],
        [4.4653e-01, 4.9135e-01, 6.2118e-02]])

In [99]:
torch.nn.functional.normalize(batch_firing_levels, p=1, dim=1)

tensor([[3.5788e-01, 6.4212e-01, 4.0343e-07],
        [1.4399e-02, 9.8560e-01, 2.8026e-45],
        [4.2575e-01, 5.7396e-01, 2.9375e-04],
        [3.8945e-01, 6.1054e-01, 9.5877e-06],
        [4.4653e-01, 4.9135e-01, 6.2118e-02]])

In [100]:
single_firing_levels = multiplication_and(single_output1_0)
''' wi = feature [i] firing level
[w1, w2, w3]
'''
single_firing_levels

tensor([0.9671, 0.9768, 0.5783])

In [101]:
#normalization should result: [0.3792, 0.2889, 0.3318]

In [102]:
torch.nn.functional.normalize(single_firing_levels, p=1, dim=0)

tensor([0.3834, 0.3873, 0.2293])

In [103]:
torch.nn.functional.normalize(single_firing_levels, p=1, dim=-1)

tensor([0.3834, 0.3873, 0.2293])

In [104]:
class FiringLevelsLayer(nn.Module):
    '''
    Class used to calculate firing levels and normalize them
    '''
    def __init__(self):
        super(FiringLevelsLayer, self).__init__()

    def forward(self, x):
        return torch.nn.functional.normalize(x.prod(dim=x.dim()-2), p=1, dim=-1)

In [105]:
layer23_0 = FiringLevelsLayer()

In [106]:
single_normalized_FL = layer23_0(single_output1_0)
''' ~wi = normalization(wi)
    wi = feature [i] firing level
[~w1, ~w2, ~w3]
'''
single_normalized_FL

tensor([0.3834, 0.3873, 0.2293])

In [107]:
batch_normalized_FL = layer23_0(batch_output1_0)
''' ~w = normalization(w)
    wi = feature [i] firing level
[[~w1, ~w2, ~w3],  (batch input1)
 [~w1, ~w2, ~w3],  (batch input2)
 ...
 [~w1, ~w2, ~w3]]  (last batch input)
'''
batch_normalized_FL

tensor([[3.5788e-01, 6.4212e-01, 4.0343e-07],
        [1.4399e-02, 9.8560e-01, 2.8026e-45],
        [4.2575e-01, 5.7396e-01, 2.9375e-04],
        [3.8945e-01, 6.1054e-01, 9.5877e-06],
        [4.4653e-01, 4.9135e-01, 6.2118e-02]])

###Testeo de pesos ponderados matlab



In [108]:
layer2_matlab = FiringLevelsLayer()

In [109]:
output_2_matlab = layer2_matlab(output_1_matlab)
output_2_matlab

tensor([[1.1257e-19, 1.5007e-01, 8.4993e-01],
        [3.9147e-15, 9.9892e-01, 1.0829e-03],
        [8.6307e-05, 9.9991e-01, 4.9215e-14],
        [1.9157e-19, 1.9100e-01, 8.0900e-01]])

###2.P Some things to consider
* **IT SHOULD BE MORE OPTIMAL TO PERFORM THE NORMALIZATION IN THE CONSEQUENT PART, that is, when the w is multiplied with the consequent function** (apparently it will be the same, however it is likely that using pytorch's functional.normalize is better )
* **WOULD IT BE USEFUL TO EXTRACT/SHOW THE FIRING LEVELS ON SCREEN WITHOUT NORMALIZING??**(should be possible in the anfis class)
* ** MUST VERIFY THAT THE NORMALIZATION USED HERE IS THE SAME ONE APPLIED IN MATLAB, the one in this jupyter notebook is a simple normalization of the type: [1, 2] -> [1/(1+2), 2/ (1+2)], the use of any exponent or special function in matlab has not been verified**

##3 Consequent Layer

###3.1 Proposal 1

In [110]:
x = torch.tensor([[1, 2],
                  [3, 4]], dtype=torch.float32)
'''
[[x1, x2],  (batch input1)
 [x1, x2]]  (batch input2)
'''

c = torch.tensor([[10, 20, 30],
                  [40, 50, 60]], dtype=torch.float32)
'''
[[p1, q1, r1],  (feature 1 consequent parameters)
 [p2, q2, r2]]  (feature 2 consequent parameters)
'''

w = torch.tensor([[100, 200],
                  [300, 400]], dtype=torch.float32)
'''
[[~w1, ~w2],  (batch input 1 normalized firing levels)
 [~w1, ~w2]]  (batch input 2 normalized firing levels)
'''

'\n[[~w1, ~w2],  (batch input 1 normalized firing levels)\n [~w1, ~w2]]  (batch input 2 normalized firing levels)\n'

In [111]:
def simple_linear(x, c):
    return x.matmul(c[:, :-1].transpose(0, 1)) + c[:, -1].unsqueeze(0)

In [112]:
'''
[[x1*p1 + x2*q1 + r1, x1*p2 + x2*q2 + r2],  (x1, x2 of batch input1)
 [x1*p1 + x2*q1 + r1, x1*p2 + x2*q2 + r2]]  (x1, x2 of batch input2)
'''
simple_linear(x, c)

tensor([[ 80., 200.],
        [140., 380.]])

In [113]:
def typical_linear(x, c, w):
    return (x.matmul(c[:, :-1].transpose(0, 1)) + c[:, -1].unsqueeze(0)).mul(w)

In [114]:
'''
[[~w1*(x1*p1 + x2*q1 + r1), ~w2*(x1*p2 + x2*q2 + r2)],  (x1, x2, ~w1 y ~w2 of batch input1)
 [~w1*(x1*p1 + x2*q1 + r1), ~w2*(x1*p2 + x2*q2 + r2)]]  (x1, x2, ~w1 y ~w2 of batch input2)
'''
typical_linear(x, c, w)

tensor([[  8000.,  40000.],
        [ 42000., 152000.]])

In [115]:
class ConsequentLayer(nn.Module):
    '''
    Class that represents the fourth layer (consequent layer) of an ANFIS

    Attributes:
        data_shape : dimension of the input data
        rules : number of rules
        function : function used as a consequent function
        consequents: array with the parameters used in each node of this layer
    '''
    def __init__(self, data_shape, rules=3, function=typical_linear):
        super(ConsequentLayer, self).__init__()
        self.rules = rules
        self.data_shape = data_shape
        self.function = function
        #CONSEQUENT PARAMETERS inicializados aleatoriamente de momento
        self.consequents = torch.rand(rules, data_shape[1] + 1)

    def forward(self, x, w):
        return self.function(x, self.consequents, w)

    def show_consequents(self):
        print("Consequents tensor:")
        print(self.consequents)

    def consequents(self):
        return self.consequents

In [116]:
consequent_layer = ConsequentLayer(train_data.shape)

In [117]:
# x : input data (for train)
train_data

tensor([[0.5234, 0.3513],
        [0.5379, 0.0069],
        [0.8262, 0.0169],
        [0.0253, 0.3757],
        [0.3157, 0.2126]])

In [118]:
# c : consequent parameters by feature
consequent_layer.consequents

tensor([[0.2836, 0.3591, 0.6421],
        [0.5865, 0.8834, 0.8654],
        [0.0703, 0.0844, 0.0985]])

In [119]:
# w: normalized firing levels for input on batch
output23

tensor([[0.6053, 0.5051, 0.6152],
        [0.6020, 0.5045, 0.6189],
        [0.6252, 0.3864, 0.6781],
        [0.6057, 0.6076, 0.5138],
        [0.5950, 0.5674, 0.5693]])

In [120]:
output4 = consequent_layer(train_data, output23)
''' ~w = normalization(w)   --> normalized firing levels
    fi = x1*pi + x2*qi + ri --> consequent function of feature i applied to an input (x1, x2)

[[~w1*f1, ~w2*f2, ~w3*f3],  (of batch input1)
 [~w1*f1, ~w2*f2, ~w3*f3],   (of batch input2)
 ...
 [~w1*f1, ~w2*f2, ~w3*f3]]   (of batch last input)
'''
output4

tensor([[0.5549, 0.7489, 0.1015],
        [0.4799, 0.5988, 0.0847],
        [0.5518, 0.5274, 0.1072],
        [0.4750, 0.7365, 0.0678],
        [0.4807, 0.7027, 0.0789]])

###3.1.O Proposal 1 Optimized and Flexible

In [121]:
single_normalized_FL

tensor([0.3834, 0.3873, 0.2293])

In [122]:
batch_normalized_FL

tensor([[3.5788e-01, 6.4212e-01, 4.0343e-07],
        [1.4399e-02, 9.8560e-01, 2.8026e-45],
        [4.2575e-01, 5.7396e-01, 2.9375e-04],
        [3.8945e-01, 6.1054e-01, 9.5877e-06],
        [4.4653e-01, 4.9135e-01, 6.2118e-02]])

In [123]:
single_data0

tensor([0.0554, 0.3409])

In [124]:
batch_data0

tensor([[0.6453, 0.5495],
        [0.1682, 0.9843],
        [0.2054, 0.4795],
        [0.0258, 0.0656],
        [0.7770, 0.1938]])

In [125]:
params = ["p", "q", "r"]
#the number of parameters depends on the dimension of the input data

consequents = torch.rand(layer1_0.rules, len(params))
'''
[[p1, q1, r1],  (consequent parameters of feature 1)
 [p2, q2, r2]]  (consequent parameters of feature 2)
 [p3, q3, r3]]  (consequent parameters of feature 3)
'''
consequents

tensor([[0.6531, 0.9190, 0.0668],
        [0.2658, 0.6789, 0.3160],
        [0.7381, 0.0850, 0.5057]])

In [126]:
def simple_linear(x, c):
    return x.matmul(c[:, :-1].transpose(0, 1)) + c[:, -1].unsqueeze(0)

In [127]:
'''fi(x1, x2) = x1*pi + x2*qi + ri, con pi, qi, ri consequent parameters of feature i

[[f1(x1, x2), f2(x1, x2), f3(x1, x2)],  ((x1, x2) of batch input1)
 [f1(x1, x2), f2(x1, x2), f3(x1, x2)]]  ((x1, x2) of batch input2)
'''
simple_linear(batch_data0, consequents)

tensor([[0.9932, 0.8606, 1.0288],
        [1.0812, 1.0289, 0.7136],
        [0.6416, 0.6961, 0.6981],
        [0.1440, 0.3674, 0.5304],
        [0.7523, 0.6541, 1.0957]])

In [128]:
'''fi(x1, x2) = x1*pi + x2*qi + ri, con pi, qi, ri consequent parameters of feature i

[f1(x1, x2), f2(x1, x2), f3(x1, x2)]  ((x1, x2) the input values)
'''
simple_linear(single_data0, consequents)

tensor([[0.4162, 0.5621, 0.5756]])

In [129]:
def weighted_linear(x, c, w):
    return (x.matmul(c[:, :-1].transpose(0, 1)) + c[:, -1]).mul(w)

In [130]:
weighted_linear(single_data0, consequents, single_normalized_FL)

tensor([0.1596, 0.2177, 0.1320])

In [131]:
weighted_linear(batch_data0, consequents, batch_normalized_FL)

tensor([[3.5545e-01, 5.5259e-01, 4.1503e-07],
        [1.5568e-02, 1.0141e+00, 1.4013e-45],
        [2.7315e-01, 3.9953e-01, 2.0507e-04],
        [5.6064e-02, 2.2431e-01, 5.0850e-06],
        [3.3594e-01, 3.2140e-01, 6.8063e-02]])

In [132]:
single_data0.shape

torch.Size([2])

In [133]:
single_data0.shape[-1]

2

In [134]:
batch_data0.shape

torch.Size([5, 2])

In [135]:
single_data0.shape[-1]

2

In [136]:
class ConsequentLayer(nn.Module):
    '''
    Class that represents the fourth layer (consequent layer) of an ANFIS

    Attributes:
        input_size : size of a single input
        rules : number of rules
        function : function used as a consequent function
        consequents: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_size, rules=3, function=weighted_linear):
        super(ConsequentLayer, self).__init__()
        self.rules = rules
        self.input_size = input_size
        self.function = function
        #CONSEQUENT PARAMETERS randomly initialized at the moment
        self.consequents = torch.rand(rules, input_size + 1)

    def forward(self, x, w):
        return self.function(x, self.consequents, w)

    @property
    def consequents_structure(self):
        print("Consequents Structure:")
        for i in range(self.rules):
            print(f"    rule {i + 1} consequent parameters: {self.consequents[i]}")

In [137]:
layer4_0 = ConsequentLayer(single_data0.shape[0])

In [138]:
single_data0

tensor([0.0554, 0.3409])

In [139]:
single_normalized_FL

tensor([0.3834, 0.3873, 0.2293])

In [140]:
layer4_0.consequents_structure

Consequents Structure:
    rule 1 consequent parameters: tensor([0.5676, 0.7368, 0.0762])
    rule 2 consequent parameters: tensor([0.3367, 0.9032, 0.2989])
    rule 3 consequent parameters: tensor([0.9027, 0.1562, 0.1817])


In [141]:
single_ponderated_consequents = layer4_0(single_data0, single_normalized_FL)
single_ponderated_consequents

tensor([0.1376, 0.2422, 0.0653])

In [142]:
batch_ponderated_consequents = layer4_0(batch_data0, batch_normalized_FL)
batch_ponderated_consequents

tensor([[3.0327e-01, 6.5016e-01, 3.4296e-07],
        [1.2915e-02, 1.2267e+00, 1.4013e-45],
        [2.3251e-01, 4.5985e-01, 1.2985e-04],
        [5.4230e-02, 2.2402e-01, 2.0640e-06],
        [2.9473e-01, 3.6143e-01, 5.6739e-02]])

###Testeo de salidas matlab

In [143]:
pc = torch.tensor([[0.6557, 0.9340, 0.7431, 0.1712, 0.2769, 0.8235],
    [0.0357, 0.6787, 0.3922, 0.7060, 0.0462, 0.6948],
    [0.8491, 0.7577, 0.6555, 0.0318, 0.0971, 0.3171]])

In [144]:
class TestingConsequentLayer(nn.Module):
    def __init__(self, input_size, rules=3, function=weighted_linear, pc=pc):
        super(TestingConsequentLayer, self).__init__()
        self.rules = rules
        self.input_size = input_size
        self.function = function
        self.consequents = pc

    def forward(self, x, w):
        return self.function(x, self.consequents, w)

    @property
    def consequents_structure(self):
        print("Consequents Structure:")
        for i in range(self.rules):
            print(f"    rule {i + 1} consequent parameters: {self.consequents[i]}")

In [145]:
layer3_matlab = TestingConsequentLayer(batch_data1.shape[1])

In [146]:
output_3_matlab = layer3_matlab(batch_data1, output_2_matlab)
output_3_matlab

tensor([[ 2.5627e-19,  2.8424e-01,  1.4158e+00],
        [ 5.9506e-15,  5.5935e-01,  1.1756e-03],
        [-2.7933e-05,  5.4990e-01, -5.0447e-14],
        [ 4.3765e-19,  1.3254e-01,  1.4349e+00]])

##4 Output Layer

###4.1 Proposal 1

In [147]:
def sum(x):
    return torch.sum(x, dim=-1)

In [148]:
sum(single_ponderated_consequents)

tensor(0.4452)

In [149]:
sum(batch_ponderated_consequents)

tensor([0.9534, 1.2396, 0.6925, 0.2783, 0.7129])

In [150]:
class OutputLayer(nn.Module):
    '''
    Class that represents the fifth layer (output layer) of an ANFIS
    '''
    def __init__(self, function=sum):
        super(OutputLayer, self).__init__()
        self.function = function

    def forward(self, x):
        return self.function(x)

###Testeo output matlab

In [151]:
layer4_matlab = OutputLayer()

In [152]:
output_4_matlab = layer4_matlab(output_3_matlab)
output_4_matlab

tensor([1.7000, 0.5605, 0.5499, 1.5674])

##5 ANFIS Structure Implementation corrections


###5.1 Proposal 1

In [153]:
#Data set
train_data = torch.rand(10,3)
x_train = train_data[:, :-1]
y_train = train_data[:, -1]

In [154]:
x_train

tensor([[0.2736, 0.9288],
        [0.9675, 0.8692],
        [0.8800, 0.8660],
        [0.2157, 0.2170],
        [0.2965, 0.3984],
        [0.2074, 0.8123],
        [0.9879, 0.4244],
        [0.5154, 0.5128],
        [0.2629, 0.8676],
        [0.4280, 0.8216]])

In [155]:
class Type3ANFIS(nn.Module):
    '''
    Class that represents a type3 ANFIS

    To initialize it:
        input_size : size of a single input
        init_rules : number of initial rules
        cf: function used as consequent function
        mf: function used as membership function
        of: function used to join the outputs of each of the rules
        mf_params: list with MF parameters names
        premises: array with the parameters used in each node of this layer
        prem_init_mode: way to initialize the premises
            =0 : to initialize them randomly
            =1 : to initialize them [MATLAB IMPLEMENTATION MUST BE IMPLEMENTED]
    '''
    def __init__(self, input_size, init_rules=3, cf=weighted_linear, mf=gaussian3, of=sum, mf_params=['sigma', 'mu', 'f'], prem_init_mode=0):
        super(Type3ANFIS, self).__init__()
        self.fuzzify_layer = FuzzifyLayer(input_size, init_rules, mf, mf_params)
        self.firing_levels_layer = FiringLevelsLayer()
        self.consequent_layer = ConsequentLayer(input_size, init_rules, cf)
        self.output_layer = OutputLayer(of)

    def forward(self, x):
        output = self.fuzzify_layer(x)
        output = self.consequent_layer(x, self.firing_levels_layer(output))
        output = self.output_layer(output)
        return output

In [156]:
anfis1 = Type3ANFIS(x_train.shape[1])

In [157]:
anfis1(x_train[0])

tensor(1.3605)

###5.2 Proposal 2

In [158]:
class Type3ANFIS(nn.Module):
    '''
    Class that represents a type3 ANFIS

    To initialize it:
        input_size : size of a single input
        init_rules : number of initial rules
        cf: function used as consequent function
        mf: function used as membership function
        of: function used to join the outputs of each of the rules
        mf_params: list with MF parameters namesr
        x_train: training data set; this parameter its used to initialize the premises uniformly,
                 if it is not entered, the premises will be initialized randomly
    '''
    def __init__(self, input_size, init_rules=3, cf=weighted_linear, mf=gaussian3, of=sum, mf_params=['sigma', 'mu', 'f'], prem_init_mode=0, x_train=[]):
        super(Type3ANFIS, self).__init__()
        self.fuzzify_layer = FuzzifyLayer(input_size, init_rules, mf, mf_params)
        if x_train != []:
            fuzzify_layer.uniform_premises(x_train)
        self.firing_levels_layer = FiringLevelsLayer()
        self.consequent_layer = ConsequentLayer(input_size, init_rules, cf)
        self.output_layer = OutputLayer(of)

    def forward(self, x):
        output = self.fuzzify_layer(x)
        output = self.consequent_layer(x, self.firing_levels_layer(output))
        output = self.output_layer(output)
        return output

In [159]:
#Data set
train_data = torch.rand(10,3)
x_train = train_data[:, :-1]
y_train = train_data[:, -1]

In [160]:
test = Type3ANFIS(x_train.shape[1])
test_batch_output = test(x_train)
test_batch_output

tensor([1.1484, 1.4095, 1.1476, 1.3311, 1.1608, 0.5426, 1.2864, 1.0021, 1.1031,
        0.8931])

In [161]:
test_single_output = test(x_train[2])
test_single_output

tensor(1.1476)

##6 Some testing and modifications

In [183]:
class FuzzifyLayer(nn.Module):
    '''
    Class that represents the first layer (fuzzification layer) of an ANFIS

    To initialize it:
        input_size : size of a single input
        init_rules : number of initial rules
        mf: function used as membership function
        params: list with parameter names
        x_train: training data set; this parameter its used to initialize the premises uniformly,
                 if it is not entered, the premises will be initialized randomly

    Other attributes:
        premises: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_size, init_rules=3, mf=gaussian2, params=['mu', 'sigma'], x_train=[]):
        super(FuzzifyLayer, self).__init__()
        self.input_size = input_size
        self.init_rules = init_rules
        self.mf = mf
        self.params = params
        if x_train == []:
            self.premises = Parameter(torch.rand(input_size, init_rules, len(params)), requires_grad=True)
        else:
            self.init_premises(x_train)

    def init_premises(self, x_train):
        self.premises = Parameter(torch.zeros(self.input_size, self.init_rules, len(self.params)), requires_grad=False)
        if self.init_rules > 1:
            min = torch.min(x_train, dim=0).values
            max = torch.max(x_train, dim=0).values
            stp = (max - min) / (self.init_rules - 1)
            for i in range(self.input_size):
                h = torch.arange(min[i], max[i] + stp[i], stp[i])
                self.premises[i, :, 0] = h[:self.init_rules]
                self.premises[i, :, 1] = stp[i]/2
                if (len(self.params) == 3): self.premises[i, :, 2] = 2
        else:
            for i in range(self.input_size):
                self.premises[i, :, 0] = torch.mean(x_train[:, i])
                self.premises[i, :, 1] = torch.std(x_train[:, i])
                if (len(self.params) == 3): self.premises[i, :, 2] = 2
        self.premises.requires_grad = True

    def forward(self, x):
        return self.mf(x.unsqueeze(x.dim()), self.premises)

    @property
    def premises_structure(self):
        print("Premises Structure:")
        for i in range(self.input_size):
            print(f"    x{i} parameters:")
            for j in range(self.init_rules):
                print(f"        rule {j + 1}:")
                [print(f"            {self.params[k]}: {self.premises[i, j, k]}") for k in range(len(self.params))]

In [177]:
x_train = torch.rand(10,3)
x_train

tensor([[0.0785, 0.8526, 0.4361],
        [0.9745, 0.2930, 0.0213],
        [0.0983, 0.3411, 0.5366],
        [0.4676, 0.2475, 0.7844],
        [0.4015, 0.1234, 0.7468],
        [0.6785, 0.8534, 0.1635],
        [0.9167, 0.9437, 0.9548],
        [0.2775, 0.7031, 0.6789],
        [0.3297, 0.1434, 0.0202],
        [0.1353, 0.8876, 0.7661]])

In [187]:
test_fuzzy_layer = FuzzifyLayer(x_train.shape[1], x_train=x_train)

In [188]:
test_fuzzy_layer.premises

Parameter containing:
tensor([[[0.0785, 0.2240],
         [0.5265, 0.2240],
         [0.9745, 0.2240]],

        [[0.1234, 0.2051],
         [0.5335, 0.2051],
         [0.9437, 0.2051]],

        [[0.0202, 0.2336],
         [0.4875, 0.2336],
         [0.9548, 0.2336]]], requires_grad=True)

In [189]:
test_fuzzy_layer.premises_structure

Premises Structure:
    x0 parameters:
        rule 1:
            mu: 0.07849198579788208
            sigma: 0.22399009764194489
        rule 2:
            mu: 0.5264722108840942
            sigma: 0.22399009764194489
        rule 3:
            mu: 0.9744523763656616
            sigma: 0.22399009764194489
    x1 parameters:
        rule 1:
            mu: 0.12335902452468872
            sigma: 0.20509079098701477
        rule 2:
            mu: 0.5335406064987183
            sigma: 0.20509079098701477
        rule 3:
            mu: 0.9437221884727478
            sigma: 0.20509079098701477
    x2 parameters:
        rule 1:
            mu: 0.0202486515045166
            sigma: 0.23362618684768677
        rule 2:
            mu: 0.48750102519989014
            sigma: 0.23362618684768677
        rule 3:
            mu: 0.9547533988952637
            sigma: 0.23362618684768677


In [170]:
class FiringLevelsLayer(nn.Module):
    '''
    Class used to calculate firing levels and normalize them
    '''
    def __init__(self):
        super(FiringLevelsLayer, self).__init__()

    def forward(self, x):
        return torch.nn.functional.normalize(x.prod(dim=x.dim()-2), p=1, dim=-1)

In [171]:
class ConsequentLayer(nn.Module):
    '''
    Class that represents the fourth layer (consequent layer) of an ANFIS

    To initialize it:
        input_size : size of a single input
        init_rules : number of init_rules
        function : function used as a consequent function

    Other attributes:
        consequents: array with the parameters used in each node of this layer
    '''
    def __init__(self, input_size, init_rules=3, function=weighted_linear):
        super(ConsequentLayer, self).__init__()
        self.init_rules = init_rules
        self.input_size = input_size
        self.function = function
        self.consequents = Parameter(torch.rand(init_rules, input_size + 1), requires_grad=True)

    def forward(self, x, w):
        return self.function(x, self.consequents, w)

    @property
    def consequents_structure(self):
        print("Consequents Structure:")
        for i in range(self.init_rules):
            print(f"    rule {i + 1} consequent parameters: {self.consequents[i]}")

In [172]:
class OutputLayer(nn.Module):
    '''
    Class that represents the fifth layer (output layer) of an ANFIS
    '''
    def __init__(self, function=sum):
        super(OutputLayer, self).__init__()
        self.function = function

    def forward(self, x):
        return self.function(x)

In [173]:
class Type3ANFIS(nn.Module):
    '''
    Class that represents a type3 ANFIS

    To initialize it:
        input_size : size of a single input
        init_rules : number of initial rules
        cf: function used as consequent function
        mf: function used as membership function
        of: function used to join the outputs of each of the rules
        mf_params: list with MF parameters namesr
        x_train: training data set; this parameter its used to initialize the premises uniformly,
                 if it is not entered, the premises will be initialized randomly
    '''
    def __init__(self, input_size, init_rules=3, cf=weighted_linear, mf=gaussian2, of=sum, mf_params=['mu', 'sigma'], x_train=[]):
        super(Type3ANFIS, self).__init__()
        self.input_size = input_size
        self.mf_params = mf_params
        if x_train == []:
            self.fuzzify_layer = FuzzifyLayer(input_size, init_rules, mf, mf_params)
        else:
            self.fuzzify_layer = FuzzifyLayer(input_size, init_rules, mf, mf_params, x_train)
        self.firing_levels_layer = FiringLevelsLayer()
        self.consequent_layer = ConsequentLayer(input_size, init_rules, cf)
        self.output_layer = OutputLayer(of)

    def forward(self, x):
        output = self.fuzzify_layer(x)
        output = self.consequent_layer(x, self.firing_levels_layer(output))
        output = self.output_layer(output)
        return output

    def firing_levels(self, x):
        x = self.fuzzify_layer(x)
        fl = x.prod(dim=x.dim()-2)
        return fl

    @property
    def premises_structure(self):
        self.fuzzify_layer.premises_structure

    @property
    def premises(self):
        return self.fuzzify_layer.premises.data

    def set_premises(self, premises):
        self.fuzzify_layer.premises = premises

    @property
    def consequents_structure(self):
        self.consequent_layer.consequents_structure

    @property
    def consequents(self):
        return self.consequent_layer.consequents.data

    def set_consequents(self, consequents):
        self.consequent_layer.consequents = consequents